# Git 브랜치와 worktree, GitHub PR 리뷰

Git은 파일을 고친 이력을 커밋 단위로 저장합니다. 한 저장소에서 다른 작업을 별도 폴더로 열 때는 worktree를 사용합니다. 뒤의 팀 실습에는 앞 단원에서 푼 예제 한 개와 그 입력·기대값·실제값을 가져오세요.

먼저 작성 중인 예제를 그대로 두고 별도 폴더에서 오타만 고칩니다. 그다음 짝이 내 변경을 읽고 새 입력을 제안합니다. 리뷰를 반영해 같은 요청을 갱신하고, 팀의 기본 버전에 두 사람의 변경을 합칩니다.

Python 3 커널과 Git이 필요합니다. Windows는 WSL 터미널에서 `git --version`을 실행하고 WSL에서 연 노트북을 사용하세요. macOS는 터미널에서 같은 명령을 씁니다. 버전이 나오면 다음 셀을 실행합니다. 명령을 찾지 못하면 [Git 설치 안내](https://git-scm.com/downloads)에서 해당 환경을 선택하세요. 아래 코드 셀은 임시 폴더만 사용하므로 혼자 반복할 수 있습니다. 두 사람의 GitHub 실습은 뒤쪽의 팀 과제와 [팀 활동지](practice.md)를 따릅니다.

### 내 결과를 다음 노트북에서 여는 준비 셀

수업 묶음의 압축을 풀고 노트북을 여세요. 이 셀은 묶음 맨 위에 `my-course` 폴더를 만듭니다. 같은 이름의 파일에 다시 저장하면 그 단계의 결과가 갱신됩니다. 뒤의 장에서 시작한다면 이 준비 셀과 해당 장의 예제를 실행하세요. 앞 장 파일이 없을 때 `read_work`의 두 번째 인자가 출발 예시가 됩니다. 낱개로 받은 노트북은 모두 같은 폴더에 둡니다.

In [ ]:
from pathlib import Path
import json

here = Path.cwd()
# 과정 폴더에서 열거나, 묶음 맨 위의 일자별 사본을 열어도 같은 장소를 씁니다.
course_root = here.parent.parent if here.parent.name == "pre-mid" else here
work = course_root / "my-course"
work.mkdir(exist_ok=True)

def save_work(name, value):
    (work / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def read_work(name, example):
    path = work / name
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return example

## 6장 · 커밋이 보관하는 변경 읽기

`git status --short`는 현재 폴더에서 바뀐 파일을 한 줄씩 표시합니다. 예를 들어 새 파일 옆의 `??`는 Git이 아직 그 파일을 추적하지 않는다는 뜻입니다. `git add 파일명`으로 다음 기록에 넣을 내용을 고르고, `git commit -m "첫 기록"`으로 기록 한 건을 남깁니다. 노트북에서는 준비 셀로 임시 저장소를 만든 뒤 `git(repo, "status")`처럼 첫 인자로 실행 폴더를 지정합니다.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess

# 새 임시 폴더 안에서만 명령을 실행합니다. 첫 셀부터 다시 실행하면 새 실습이 시작됩니다.
session = TemporaryDirectory()
root = Path(session.name)
repo = root / "team"
fix = root / "spelling"
repo.mkdir()

def git(folder, *args):
    # -C 뒤에 작업 폴더를 넘기면, 다른 저장소에서 명령을 실행할 염려가 없습니다.
    result = subprocess.run(["git", "-C", str(folder), *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout.rstrip()

git(repo, "init", "-b", "main")  # main을 첫 브랜치 이름으로 정합니다.
git(repo, "config", "user.name", "Classroom")
git(repo, "config", "user.email", "classroom@example.invalid")
git(repo, "config", "commit.gpgsign", "false")  # 연습 저장소의 커밋에는 서명을 사용하지 않습니다.
guide = repo / "cases.md"
_ = guide.write_text("문벙 예제에서는 값을 하나 바꿔 보세요.\n", encoding="utf-8")
git(repo, "add", "cases.md")
git(repo, "commit", "-m", "첫 실습 안내 추가")  # -m 뒤에 이 커밋의 설명을 씁니다.
print(git(repo, "log", "-1", "--format=%s"))

첫 실습 안내 추가


`첫 실습 안내 추가`는 방금 저장한 커밋의 설명입니다. `log`는 기록을 읽고, `-1`은 가장 최근 한 건을 고르며, `--format=%s`는 제목만 보여 줍니다. 이후의 `git(repo, "status")`는 터미널에서 이 폴더를 열고 `git status`를 쓴 것과 같습니다.

### git switch -c로 브랜치 만들고 이동하기

**브랜치**는 이어서 커밋할 작업의 이름입니다. 팀의 기준 버전은 `main`에 두고, 내 예제는 새 브랜치에 작성합니다. `switch -c` 뒤에 이름을 적으면 현재 커밋에서 브랜치를 만들고 그곳으로 이동합니다.

`git(repo, "switch", "-c", "case/min")`은 연습 폴더에서 `git switch -c case/min`을 실행합니다. `-c`는 새 브랜치를 만들라는 옵션입니다. 작업 폴더는 그대로이고, 다음 커밋이 기록될 브랜치가 바뀝니다. 이어서 파일을 만들면 `git status --short`에서 아직 추적하지 않는 파일 앞에 `??`가 보입니다.

### 앞 장의 검수 사례를 브랜치로 가져오기

문법 과정은 A, 디자인 패턴 과정은 B를 사용하세요. 앞 장 파일이 없을 때는 선택한 경로의 완성 사례로 시작합니다.

두 경로의 파일이 모두 있으면 기본 선택은 A입니다. B로 진행하려면 다음 셀의 첫 줄을 `route = "B"`로 바꾸세요.

In [ ]:
route = "B" if (work / "git-case-b.json").exists() and not (work / "git-case-a.json").exists() else "A"
examples = {
    "A": {"input": [124, 30], "expected": 154, "observed": 154},
    "B": {"input": " 우산 ", "expected": ["[우산]", ["우산"]], "observed": ["[우산]", ["우산"]]},
}
chosen = read_work("git-case-" + route.lower() + ".json", examples[route])
case_text = json.dumps(chosen, ensure_ascii=False, indent=2)
print(chosen["expected"] == chosen["observed"])

True


기본 예에서는 True가 나옵니다. False라면 동료도 같은 차이를 볼 수 있도록 기대값을 그대로 두고 현재 결과를 저장하세요. 다음 셀이 이 JSON을 `min-case.md`에 씁니다.

In [2]:
git(repo, "switch", "-c", "case/min")
draft = repo / "min-case.md"
_ = draft.write_text(case_text, encoding="utf-8")
print(git(repo, "status", "--short", "--branch"))  # 현재 브랜치와 변경 파일을 함께 봅니다.

## case/min
?? min-case.md


`## case/min`은 현재 브랜치입니다. 그 아래 `?? min-case.md`는 <mark>디스크에 파일이 생겼지만 Git이 아직 추적하지 않는다는 뜻입니다.</mark> 여기서 커밋하기에는 설명이 덜 쓰였습니다. 그런데 동료가 기준 버전의 ‘문벙’이라는 오타를 발견했습니다.

### git worktree add로 두 번째 작업 폴더 열기

터미널에서는 `git worktree add -b fix/min-type 새폴더 main`으로 실행합니다. 명령을 마치면 원래 폴더는 그대로 열려 있고 새 폴더에는 `main`에서 시작한 `fix/min-type` 브랜치의 파일이 보입니다.

**워크트리(worktree)**는 어느 브랜치의 파일을 실제로 펼쳐 놓은 작업 폴더입니다. 한 저장소에 두 번째 워크트리를 만들면 두 브랜치의 파일을 서로 다른 창에서 열 수 있습니다. 아래에서 `-b` 뒤는 새 브랜치 이름, 그다음은 새 폴더 경로, 마지막 `main`은 출발할 커밋을 가리킵니다.

In [3]:
git(repo, "worktree", "add", "-b", "fix/min-type", str(fix), "main")
print(git(fix, "status", "--short", "--branch"))

## fix/min-type


두 번째 폴더에는 `## fix/min-type`만 보입니다. 첫 폴더에서 작성 중인 파일은 아직 커밋에 들어 있지 않아서 `main`에서 연 두 번째 폴더에 없습니다. 이 폴더의 안내문만 고쳐 봅시다.

In [4]:
other_guide = fix / "cases.md"
before = other_guide.read_text(encoding="utf-8")
_ = other_guide.write_text(before.replace("문벙", "문법"), encoding="utf-8")
print(git(fix, "diff", "--", "cases.md"))  # 수정한 파일을 커밋하기 전에 비교합니다.

diff --git a/cases.md b/cases.md
index 178bb98..c207984 100644
--- a/cases.md
+++ b/cases.md
@@ -1 +1 @@
-문벙 예제에서는 값을 하나 바꿔 보세요.
+문법 예제에서는 값을 하나 바꿔 보세요.


파일 이름 아래에서 `-`로 시작하는 줄은 고치기 전 내용이고, `+`로 시작하는 줄은 고친 뒤 내용입니다. ‘문벙’이 ‘문법’으로 바뀌었는지 두 줄을 읽어 보세요. 이제 이 파일을 다음 커밋에 넣습니다.

In [5]:
git(fix, "add", "cases.md")
print(git(fix, "diff", "--cached", "--name-only"))  # 다음 커밋에 들어갈 파일 이름만 표시합니다.

cases.md


목록에는 `cases.md`가 한 번 나옵니다. 여기에 `min-case.md`까지 보인다면 지금 어느 폴더의 변경을 골랐는지 확인하세요.

In [6]:
git(fix, "commit", "-m", "문법 실습 안내의 오타 수정")
print(git(fix, "log", "-1", "--format=%s"))

문법 실습 안내의 오타 수정


`fix/min-type` 브랜치의 최신 기록은 오타 수정입니다. 첫 폴더에는 어떤 상태가 남았을까요?

### 직접 확인 · 커밋 제목과 브랜치 이름 구별하기

첫 폴더의 현재 브랜치와 미커밋 파일을 출력하세요. 다음 줄에는 두 번째 폴더의 최신 커밋에 들어간 파일 이름만 출력하세요. 최신 커밋을 가리키는 이름은 `HEAD`입니다. `git show --format= --name-only HEAD`에서 빈 `--format=`은 제목 등의 출력을 생략합니다.

In [7]:
# repo에서 git status --short --branch를 실행해 브랜치와 작성 중인 파일을 읽으세요.
# fix에서는 git show --format= --name-only HEAD로 최근 커밋에 든 파일을 출력하세요.
# 노트북의 git(폴더, 명령, 옵션, …) 함수에는 각 옵션을 별도 문자열로 넘기세요.

<details><summary>확인 명령과 실행 결과 보기</summary>
<pre><code># 첫 폴더에서 예제 초안이 그대로 있는지 확인합니다.
print(git(repo, &quot;status&quot;, &quot;--short&quot;, &quot;--branch&quot;))
# 둘째 폴더에서 방금 기록한 파일만 확인합니다.
print(git(fix, &quot;show&quot;, &quot;--format=&quot;, &quot;--name-only&quot;, &quot;HEAD&quot;))</code></pre><pre>## case/min
?? min-case.md
cases.md</pre><p>첫 폴더에는 작성 중인 min-case.md가 남아 있습니다. 둘째 폴더의 최신 커밋에는 cases.md만 들어 있습니다. 동료에게 오타 수정을 먼저 보낼 브랜치는 fix/min-type입니다.</p>
</details>

### 한 파일에 수정 두 번 · 커밋에는 어느 문장이 들어갈까요?

메모를 고쳐 저장한 뒤 git add를 하고, 그 메모를 다시 고쳤습니다. 커밋 대기 영역(스테이지)에는 언제의 문장이 있을까요? 작업 폴더의 최신 내용, add로 골라 둔 내용, 마지막 커밋의 내용을 세 장의 원고처럼 구분하면 diff를 읽기 쉽습니다. 앞에서 연 임시 오타 폴더에 새 인계 메모를 만들어 둘 사이의 차이를 보겠습니다.

다음 셀에서는 61초를 스테이징한 뒤 작업 폴더의 파일만 60초로 고칩니다. 첫 출력의 AM은 새 파일을 스테이징한 뒤 다시 수정했다는 표시입니다. 두 번째 명령의 `git show :handoff.md`는 커밋할 사본을, 마지막 줄의 `read_text()`는 작업 폴더의 파일을 읽습니다. 실행한 뒤 `print(git(fix, "diff", "--", "handoff.md"))`를 새 셀에 써서 스테이징 이후에 바꾼 한 줄을 확인하세요. `diff` 뒤에 `"--cached"`를 추가하면 마지막 커밋과 스테이징 영역을 비교합니다.

In [8]:
handoff = fix / "handoff.md"
_ = handoff.write_text("입력 61초\n", encoding="utf-8")
git(fix, "add", "handoff.md")  # 61초 문장을 다음 커밋에 담습니다.
_ = handoff.write_text("입력 60초\n", encoding="utf-8")
print(git(fix, "status", "--porcelain", "--", "handoff.md"))
print(git(fix, "show", ":handoff.md"))  # 맨 앞 콜론은 스테이지에 담긴 파일을 가리킵니다.
print(handoff.read_text(encoding="utf-8").strip())

AM handoff.md
입력 61초
입력 60초


AM에서 첫 칸 A는 새 파일을 커밋 대상으로 고른 상태, 둘째 칸 M은 고른 뒤 작업 폴더에서 다시 고친 상태입니다. 파일에는 60초, 지금 커밋할 내용에는 61초가 있습니다. 실행 결과가 최신인데 PR 본문이 옛 사례를 가리키는 이유가 여기에 있을 수 있습니다.

### 개인 훈련 · 필요한 내용만 골라 교대 기록 남기기

상대가 받아야 할 것은 60초짜리 인계 메모 한 개입니다. 임시 오타 폴더에서 handoff.md를 다시 선택해 커밋하세요. 최근 커밋의 **파일 본문**을 읽어 숫자를 검증하고 첫 폴더의 미커밋 초안도 확인하세요. 잘못된 숫자를 발견해도 워크트리 전체를 지우지 않고 어느 단계에서 고칠지 결정하세요.

In [9]:
# fix에서 handoff.md만 골라 커밋하고 HEAD:handoff.md로 확정된 문장을 읽습니다.
# repo에서는 case/min의 미커밋 초안이 보이는지 확인합니다.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>git(fix, &quot;add&quot;, &quot;handoff.md&quot;)
git(fix, &quot;commit&quot;, &quot;-m&quot;, &quot;확인한 60초 인계 입력 기록&quot;)
print(git(fix, &quot;show&quot;, &quot;HEAD:handoff.md&quot;))  # 커밋된 실제 내용
print(git(repo, &quot;status&quot;, &quot;--short&quot;, &quot;--branch&quot;))</code></pre>
<pre>입력 60초
## case/min
?? min-case.md</pre>
<p>스테이지를 최신 문장으로 갱신했기 때문에 마지막 커밋에는 60초가 남습니다. 원래 폴더에서는 case/min과 min-case.md가 그대로 보입니다. add 전이라면 파일을 수정하고, add 후라면 재선택하고, 이미 동료에게 전달했다면 고치는 새 커밋을 만들어 변경을 추적합니다.</p></details>

### 초안과 별도 작업 폴더의 변경 구분하기

6장에서 고친 오타와 선택 경로의 예제는 서로 다른 작업 폴더에 있습니다. 다음 장에서 동료가 볼 검수 자료를 따로 보관하세요.

In [ ]:
submitted = json.loads(draft.read_text(encoding="utf-8"))
save_work("git-submission.json", submitted)
print(submitted == chosen)

True


기본 예에서 True는 브랜치의 초안과 5장 결과의 세 항목이 모두 같다는 뜻입니다. 아직 GitHub에 올린 것은 없습니다.

## 7장 · 팀에 보내서 같은 PR에서 수정하기

`git clone 저장소주소`는 팀 저장소의 이력을 내 컴퓨터에 복제합니다. `origin`은 복제한 원격 저장소의 기본 이름입니다. 내 브랜치가 `fix-typo`일 때 `git push -u origin fix-typo`를 실행하면 그 브랜치의 커밋이 원격으로 올라갑니다. GitHub에서 이 브랜치를 팀 브랜치에 합쳐 달라고 보내는 요청이 Pull Request(PR)입니다.

팀 실습은 둘이 한 조로 진행합니다. [시작 문서](cases.md)에는 각자 고칠 오타가 하나씩 있습니다. [팀 활동지](practice.md)에서 팀 저장소를 준비하고, 워크트리 실험을 자기 Python 예제로 다시 해 보세요. 이어서 오타와 예제를 각각 별도 PR로 보냅니다. 아래 질문을 먼저 풀고 실제 주소와 커밋을 활동지에 남기세요.

### git fetch로 origin/main 갱신하고 로컬 main과 비교하기

`git fetch origin`은 원격 커밋을 받아 내 저장소의 `origin/main`을 갱신합니다. 현재 브랜치 `main`과 그 작업 파일은 이 명령만으로 바뀌지 않습니다. `git log -1 --oneline origin/main`과 `git log -1 --oneline main`으로 각각 가리키는 커밋을 확인하세요. 아래 실험은 임시 저장소 두 개로 동료의 push를 재현합니다.

6장을 건너뛰었다면 맨 위의 파일 준비 셀, 6장의 임시 저장소를 만드는 셀(git-03), A·B 사례를 고르는 셀(git-select-case)을 차례로 실행하세요. `git-submission.json`을 받지 못한 경우 아래 코드는 선택 경로의 완성 예시를 사용합니다.

In [10]:
shared_location = root / "classroom.git"
git(root, "init", "--bare", "-b", "main", str(shared_location))  # 커밋을 교환할 저장소입니다.
git(repo, "push", str(shared_location), "main")
viewer = root / "reader"
writer = root / "writer"
git(root, "clone", str(shared_location), str(viewer))
git(root, "clone", str(shared_location), str(writer))
git(writer, "config", "user.name", "Classmate")
git(writer, "config", "user.email", "classmate@example.invalid")
git(writer, "config", "commit.gpgsign", "false")
_ = (writer / "review.md").write_text("61초면 예약 2분", encoding="utf-8")
reviewed_case = read_work("git-submission.json", chosen)
_ = (writer / "case.json").write_text(json.dumps(reviewed_case, ensure_ascii=False), encoding="utf-8")
git(writer, "add", "review.md", "case.json")
git(writer, "commit", "-m", "61초 경계값 검수")
git(writer, "push", "origin", "main")
print(git(viewer, "log", "-1", "--format=%s", "origin/main"))

첫 실습 안내 추가


동료가 push를 마쳤는데도 내 복사본에서 읽은 origin/main의 제목은 ‘첫 실습 안내 추가’입니다. 내 컴퓨터가 아직 새 기록을 받아 보지 않았기 때문입니다. 표시된 커밋 제목이 같은지보다는 **어느 폴더의 어느 이름을 읽었는지**를 짚어 보세요.

`git fetch origin`이 끝나면 로컬에서 읽는 원격 브랜치 기록 `origin/main`이 갱신됩니다. 지금 작업하는 `main`과 파일 내용은 그대로일 수 있습니다. 아래 두 `log`는 같은 폴더에서 서로 다른 브랜치 끝의 커밋 제목을 한 개씩 읽습니다. 제목이 다르면 다음 실습에서 어느 쪽을 반영할지 알 수 있습니다.

In [11]:
git(viewer, "fetch", "origin")  # 팀에 올라온 새 기록을 받아옵니다.
print(git(viewer, "log", "-1", "--format=%s", "origin/main"))
print(git(viewer, "log", "-1", "--format=%s", "main"))
print((viewer / "review.md").exists())  # 내 작업 폴더에도 벌써 새 파일이 생겼을까요?

61초 경계값 검수
첫 실습 안내 추가
False


origin/main은 동료의 61초 검수 커밋으로 이동했습니다. 내 main과 작업 폴더는 그대로입니다. 내 커밋이 따로 없고 파일 수정도 없는 이 예에서는 새 커밋까지 앞으로 이동하는 방식으로 반영할 수 있습니다.

### 검수 실습 · 파일이 보이지 않는 원인을 좁히기

동료가 ‘올렸습니다’라고 말했지만 내 편집기에는 review.md가 없습니다. 위 두 커밋 제목에서 원인을 찾아내고 최신 기록을 내 main에 반영하세요. 재현 결과를 말할 때는 파일이 생겼다는 사실과 문서 내용이 기대와 맞는지를 함께 확인합니다.

In [12]:
# viewer의 main에 origin/main을 반영한 뒤 review.md의 본문을 출력하세요.
# git(viewer, "log", "-1", "--format=%s")로 반영 뒤 커밋 제목도 확인하세요.

<details><summary>모범 풀이와 실행 결과 보기</summary>
<pre><code>git(viewer, &quot;merge&quot;, &quot;--ff-only&quot;, &quot;origin/main&quot;)  # 새 커밋까지 내 main을 앞으로 옮깁니다.
print((viewer / &quot;review.md&quot;).read_text(encoding=&quot;utf-8&quot;))
print(git(viewer, &quot;status&quot;, &quot;--short&quot;, &quot;--branch&quot;))
print(git(viewer, &quot;log&quot;, &quot;-1&quot;, &quot;--format=%s&quot;))</code></pre>
<pre>61초면 예약 2분
## main...origin/main
61초 경계값 검수</pre>
<p>확인한 입력은 61초이고 예약 시간은 2분입니다. 새 파일 본문과 main...origin/main 상태를 함께 확인했습니다. 실제 짝 실습에서 이 명령이 앞쪽으로만 이동할 수 없다고 알리면 로컬과 원격 양쪽에 새 기록이 있을 수 있습니다. 지우는 명령부터 쓰지 말고 log로 각 기록과 변경 내용을 읽어 보세요.</p></details>

### 팀 과제 1 · 어느 브랜치에서 리뷰를 반영할까요?

짝이 내 예제를 읽고 “입력을 하나 바꾸면 어떤 결과가 나오는지 보여 주세요”라고 했습니다. 내 예제 PR이 열린 상태입니다. 앞 단원의 코드를 새 입력으로 실행하고 그 결과를 자기 예제 기록에 추가하세요. 어느 폴더에서 어떤 파일을 기록하고 어디로 보내야 할까요? 짝은 기존 PR의 파일 차이에서 새 입력과 출력을 찾아야 합니다.

<details><summary>민의 명령과 리뷰 답변 예시 보기</summary>
<pre><code># team 폴더에서 현재 브랜치가 case/min인지 확인합니다.
git branch --show-current
git add min/case.ipynb min/cases.md
git commit -m &quot;0초 입력을 실행하고 결과를 기록&quot;
git push origin case/min</code></pre><pre>입력 0시간 0분 0초 / 기대값 0초 / 실제값 0초</pre><p>학생이 팀에서 남길 기록의 예입니다. 같은 원격 브랜치에 올렸으므로 기존 PR에 수정 커밋이 추가됩니다. 짝에게 변경한 셀과 케이스를 짚어 답글을 남기세요. 실제 파일 이름과 값은 자신이 실행한 예제를 따릅니다.</p>
</details>

### 팀 과제 2 · 같은 안내 줄을 둘 다 고치면?

민과 윤이 같은 기본 커밋에서 루트 `cases.md`의 ‘확인할 것’ 한 줄을 다르게 고쳤습니다. 민의 변경이 먼저 병합되면 Git은 윤의 줄과 민의 줄을 자동으로 합칠 수 없습니다. 이 상태가 **충돌**입니다. 두 사람이 추가한 조건을 모두 살려 한 줄로 남기세요. 팀에서 일어난 충돌을 활동지 순서대로 해결한 뒤 원래 예제와 동료 예제를 다시 실행합니다.

<details><summary>충돌한 줄과 합친 결과 예시 보기</summary>
<pre><code>&lt;&lt;&lt;&lt;&lt;&lt;&lt; HEAD
확인할 것: 입력과 실제값, 실행 순서
=======
확인할 것: 입력과 실제값, 단위
&gt;&gt;&gt;&gt;&gt;&gt;&gt; origin/main</code></pre><pre>확인할 것: 입력과 실제값, 단위, 실행 순서</pre><p>윤이 자기 브랜치에서 origin/main을 합친 경우입니다. 위쪽에는 현재 브랜치의 줄, 아래쪽에는 가져온 줄이 보입니다. 두 요구를 한 줄로 쓰고 경계 표식 세 줄을 모두 지운 뒤 Git에 해결한 파일을 등록하세요.</p>
</details>

### 팀 과제 3 · 짝의 컴퓨터에서 다시 실행하기

양쪽 PR이 병합된 후 짝은 자신의 기본 브랜치를 갱신합니다. 내 예제 노트북을 새 커널로 열어 기록에 적힌 첫 입력과 리뷰 때 추가한 입력을 각각 실행해 보세요. 셀을 실행할 때 필요한 변수가 들어 있는지, 실제 출력이 예제 기록과 맞는지 확인하세요. 결과가 다르면 입력과 실제 출력을 PR에 남깁니다.

<details><summary>다시 실행한 동료의 기록 예시 보기</summary>
<pre><code># 갱신한 team 폴더에서 예제 셀의 파일과 마지막 변경을 읽습니다.
git log -1 --format=%s -- min/case.ipynb</code></pre><pre>0초 입력을 실행하고 결과를 기록</pre><p>실습용 커밋 제목의 예입니다. 이 제목만으로 셀의 실행을 확인할 수는 없습니다. 짝이 노트북을 직접 실행하고 ‘첫 입력 3723초, 새 입력 0초를 재현했습니다’처럼 자신이 본 값을 남겨야 합니다.</p>
</details>

### 리뷰를 받아들일 때 원래 요구가 사라지지 않았는지 보기

리뷰어는 ‘60초일 때도 2분이 나오나요?’라고 묻습니다. 원래 요구는 1분을 **조금이라도 초과했을 때** 다음 분을 추가하는 것이었습니다. 이 경우 좋은 리뷰 답변은 동의하는 말보다 59초, 60초, 61초를 직접 실행한 값입니다. 60초에서 요구와 다르면 코드를 수정하고, 요구와 맞다면 기대값을 적어 설명합니다. 리뷰를 반영했다고 표시하기 전에 어떤 파일과 어느 커밋에 근거가 남았는지 찾아보세요.

**혼자 해 볼 판단.** 팀원이 두 변경을 함께 올렸습니다. 하나는 “입력 60초 → 예약 1분”의 재현 사례이고, 하나는 “모든 녹음에 여유 1분 추가”입니다. 원래 운영 규칙은 60초면 1분, 61초면 2분입니다. 변경한 줄에서 승인할 것과 논의를 돌려보낼 것을 골라 실제 PR에 남길 리뷰 두 문장을 쓰세요. 상대가 코드를 고친 뒤에는 이전에 맞던 61초 사례까지 실행합니다.

<details><summary>판단과 리뷰 예시 보기</summary><p>“60초를 1분으로 처리한 사례는 규칙에 맞습니다. 여유 1분을 항상 더하면 60초가 2분이 되는데, 새 운영 규칙이 합의된 것인지 확인해 주세요.” 재검토에는 60초 → 1분, 61초 → 2분의 두 입력을 함께 남깁니다. 확인한 커밋이 PR의 마지막 수정과 다르면 마지막 커밋에서 재실행합니다.</p></details>

### 동료의 커밋에서 같은 사례 다시 열기

로컬 원격으로 주고받은 커밋에서 case.json을 읽습니다. 실습에서 받은 커밋과 동료가 담은 입력이 일치하는지 확인하고 나서 팀 GitHub PR에서도 같은 파일을 비교하세요.

In [ ]:
git(viewer, "merge", "--ff-only", "origin/main")
from_peer = json.loads((viewer / "case.json").read_text(encoding="utf-8"))
print(from_peer == reviewed_case)
print(from_peer["expected"] == from_peer["observed"])

True
True


기본 사례의 두 값은 True와 True입니다. 첫 줄은 같은 자료를 받았는지, 둘째 줄은 그 자료가 기록한 동작이 기대와 맞았는지 알려 줍니다. 코드 자체를 다시 시험하려면 함께 제출한 노트북에서 이 입력을 실행하세요.

## 다음 실습에서 가져갈 것

활동지에 자신이 올린 예제 PR, 동료에게 남긴 리뷰와 반영한 커밋을 기록하세요. 다음 단원에서는 모델에게 보낸 입력과 돌아온 응답을 구분합니다. 이번에 만든 팀 저장소에서 새 작업 브랜치를 여는 방법과, 동료가 다시 실행할 수 있도록 입력·기대값·실제값을 남긴 방식을 그대로 쓸 수 있습니다.